### Import Libraries

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_score, f1_score,recall_score

C:\Users\sahas\AppData\Roaming\Python\Python313\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


### Load Dataset

In [2]:
df = pd.read_csv("US_Accidents_March23.csv", nrows=1000000)  

In [3]:
df.shape

(1000000, 46)

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 46 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   ID                     1000000 non-null  str    
 1   Source                 1000000 non-null  str    
 2   Severity               1000000 non-null  int64  
 3   Start_Time             1000000 non-null  str    
 4   End_Time               1000000 non-null  str    
 5   Start_Lat              1000000 non-null  float64
 6   Start_Lng              1000000 non-null  float64
 7   End_Lat                0 non-null        float64
 8   End_Lng                0 non-null        float64
 9   Distance(mi)           1000000 non-null  float64
 10  Description            999999 non-null   str    
 11  Street                 998288 non-null   str    
 12  City                   999972 non-null   str    
 13  County                 1000000 non-null  str    
 14  State                  1000000

In [5]:
df.sample(5)

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),...,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
142165,A-142172,Source2,2,2016-06-28 08:19:14,2016-06-28 09:04:14,28.177673,-82.664536,NaN,NaN,0.00,...,False,False,False,False,True,False,Day,Day,Day,Day
571082,A-575542,Source2,2,2022-07-05 06:29:53,2022-07-05 07:29:17,35.077080,-81.725563,NaN,NaN,0.00,...,False,False,False,False,False,False,Day,Day,Day,Day
114298,A-114305,Source2,3,2016-06-12 04:57:58,2016-06-12 05:27:58,34.029945,-118.442192,NaN,NaN,0.00,...,False,False,False,False,False,False,Night,Night,Day,Day
700777,A-710469,Source2,2,2022-02-14 15:08:12,2022-02-14 16:07:50,33.854511,-84.492783,NaN,NaN,0.24,...,False,False,False,False,True,False,Day,Day,Day,Day
659404,A-669084,Source2,2,2022-03-22 04:45:00,2022-03-22 05:44:55,29.784309,-95.436394,NaN,NaN,0.00,...,False,False,False,False,False,False,Night,Night,Night,Night


In [6]:
df.isnull().sum()

ID                             0
Source                         0
Severity                       0
Start_Time                     0
End_Time                       0
Start_Lat                      0
Start_Lng                      0
End_Lat                  1000000
End_Lng                  1000000
Distance(mi)                   0
Description                    1
Street                      1712
City                          28
County                         0
State                          0
Zipcode                      136
Country                        0
Timezone                     451
Airport_Code                1432
Weather_Timestamp           9927
Temperature(F)             15298
Wind_Chill(F)             460589
Humidity(%)                16553
Pressure(in)               12376
Visibility(mi)             18931
Wind_Direction             13570
Wind_Speed(mph)           100398
Precipitation(in)         475316
Weather_Condition          18211
Amenity                        0
Bump      

### Data Cleaning

In [7]:
# Drop unwanted column
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

# Separate numeric and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Fill numerical columns with mean
for col in num_cols:
    df[col] = df[col].fillna(df[col].mean())

# Fill categorical columns with mode
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("After cleaning:", df.shape)

C:\Users\sahas\AppData\Local\Temp\ipykernel_10564\3227519.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object']).columns


After cleaning: (1000000, 46)


In [8]:
df.shape

(1000000, 46)

In [9]:
df.isnull().sum()

ID                             0
Source                         0
Severity                       0
Start_Time                     0
End_Time                       0
Start_Lat                      0
Start_Lng                      0
End_Lat                  1000000
End_Lng                  1000000
Distance(mi)                   0
Description                    0
Street                         0
City                           0
County                         0
State                          0
Zipcode                        0
Country                        0
Timezone                       0
Airport_Code                   0
Weather_Timestamp              0
Temperature(F)                 0
Wind_Chill(F)                  0
Humidity(%)                    0
Pressure(in)                   0
Visibility(mi)                 0
Wind_Direction                 0
Wind_Speed(mph)                0
Precipitation(in)              0
Weather_Condition              0
Amenity                        0
Bump      

In [10]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 46 columns):
 #   Column                 Non-Null Count    Dtype  
---  ------                 --------------    -----  
 0   ID                     1000000 non-null  str    
 1   Source                 1000000 non-null  str    
 2   Severity               1000000 non-null  int64  
 3   Start_Time             1000000 non-null  str    
 4   End_Time               1000000 non-null  str    
 5   Start_Lat              1000000 non-null  float64
 6   Start_Lng              1000000 non-null  float64
 7   End_Lat                0 non-null        float64
 8   End_Lng                0 non-null        float64
 9   Distance(mi)           1000000 non-null  float64
 10  Description            1000000 non-null  str    
 11  Street                 1000000 non-null  str    
 12  City                   1000000 non-null  str    
 13  County                 1000000 non-null  str    
 14  State                  1000000

### Feature Engineering

In [11]:
if 'Start_Time' in df.columns:
    df['Start_Time'] = pd.to_datetime(df['Start_Time'])

    df['year'] = df['Start_Time'].dt.year
    df['month'] = df['Start_Time'].dt.month
    df['hour'] = df['Start_Time'].dt.hour

    df.drop('Start_Time', axis=1, inplace=True)

In [12]:
df.shape

(1000000, 48)

### Encoding Categorical Columns

In [13]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = le.fit_transform(df[col])

C:\Users\sahas\AppData\Local\Temp\ipykernel_10564\3391690625.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:


### X and y Split

In [14]:
from sklearn.model_selection import train_test_split
target_column = 'Severity'   

X = df.drop(target_column, axis=1)
y = df[target_column]

print(X.shape, y.shape)

(1000000, 47) (1000000,)


In [15]:
print("df shape:", df.shape)
print("X shape:", X.shape)
print("y shape:", y.shape)

df shape: (1000000, 48)
X shape: (1000000, 47)
y shape: (1000000,)


### Train-Test Split

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=27
)

print(X_train.shape, X_test.shape)

(800000, 47) (200000, 47)


In [17]:
X_train.value_counts

<bound method DataFrame.value_counts of             ID  Source  End_Time  Start_Lat   Start_Lng  End_Lat  End_Lng  \
430917  377463       0    400305  30.249893  -97.724922      NaN      NaN   
232059  156509       0    261178  39.825405  -75.083199      NaN      NaN   
344084  280981       0    333551  33.642651 -117.733383      NaN      NaN   
510625  466027       0    497349  30.243336  -97.763153      NaN      NaN   
664205  637743       0    824590  32.459812  -86.387772      NaN      NaN   
...        ...     ...       ...        ...         ...      ...      ...   
196408  116896       0    252788  40.772182  -73.625504      NaN      NaN   
539167  498005       1    946517  33.988899 -118.397324      NaN      NaN   
560968  522368       1    925245  42.234077  -88.814163      NaN      NaN   
380600  321555       0    352141  40.318649  -75.782585      NaN      NaN   
791571  779260       0    701706  39.996731  -76.676277      NaN      NaN   

        Distance(mi)  Description  

In [18]:
X_test.value_counts

<bound method DataFrame.value_counts of             ID  Source  End_Time  Start_Lat   Start_Lng  End_Lat  End_Lng  \
613855  581644       0    873615  40.696617  -74.218597      NaN      NaN   
946001  950850       0    553156  37.972530 -122.039337      NaN      NaN   
719848  699566       0    770821  34.053616 -117.781311      NaN      NaN   
588158  552834       0    898580  41.904335  -88.032494      NaN      NaN   
808725  798320       0    684799  34.856079  -82.267197      NaN      NaN   
...        ...     ...       ...        ...         ...      ...      ...   
488200  441110       0    468916  29.940050  -95.303795      NaN      NaN   
820357  811245       0    674162  34.872131  -82.224396      NaN      NaN   
417517  362574       0    421322  42.457096  -87.825310      NaN      NaN   
325026  259806       0    309704  43.193573  -83.660622      NaN      NaN   
717906  697407       0    773208  39.781811  -76.959007      NaN      NaN   

        Distance(mi)  Description  

In [19]:
y_train.value_counts

<bound method IndexOpsMixin.value_counts of 430917    2
232059    3
344084    3
510625    2
664205    3
         ..
196408    2
539167    1
560968    3
380600    2
791571    2
Name: Severity, Length: 800000, dtype: int64>

In [20]:
y_test.value_counts

<bound method IndexOpsMixin.value_counts of 613855    2
946001    2
719848    2
588158    1
808725    3
         ..
488200    2
820357    2
417517    2
325026    2
717906    2
Name: Severity, Length: 200000, dtype: int64>

### Feature Scaling

In [21]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

C:\Users\sahas\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\extmath.py:1207: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
C:\Users\sahas\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\extmath.py:1212: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
C:\Users\sahas\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\extmath.py:1236: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


### Model Training

# Decision Tree

In [56]:
model_dt = DecisionTreeClassifier(
    max_depth=18,          
    random_state=27
)

# Train the model
model_dt.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",18
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",27
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current n

In [57]:
y_pred_dt = model_dt.predict(X_test)

In [58]:
acc_dt = accuracy_score(y_test, y_pred_dt)
recall_dt = recall_score(y_test, y_pred_dt, average='macro')

print("Accuracy (DT):", acc_dt)
print("Recall (DT - macro):", recall_dt)

print("\nCLASSIFICATION REPORT (DT)\n")
print(classification_report(y_test, y_pred_dt))

print("\nCONFUSION MATRIX (DT)\n")
print(confusion_matrix(y_test, y_pred_dt))

Accuracy (DT): 0.920785
Recall (DT - macro): 0.7216468025950358

CLASSIFICATION REPORT (DT)

              precision    recall  f1-score   support

           1       0.85      0.83      0.84      7722
           2       0.94      0.94      0.94    121413
           3       0.90      0.91      0.90     70264
           4       0.35      0.22      0.27       601

    accuracy                           0.92    200000
   macro avg       0.76      0.72      0.74    200000
weighted avg       0.92      0.92      0.92    200000


CONFUSION MATRIX (DT)

[[  6372   1157    191      2]
 [  1009 113963   6386     55]
 [   110   6275  63692    187]
 [     5     75    391    130]]


In [59]:
precision_dt=precision_score(y_test, y_pred_dt, average='macro')

In [60]:
f1_dt=f1_score(y_test, y_pred_dt, average='macro')

#### Decision Tree (Class balance)

In [70]:
model_dt_bal = DecisionTreeClassifier(
    max_depth=30,
    min_samples_split=10,
    class_weight='balanced',
    random_state=27
)
model_dt_bal.fit(X_train, y_train)

,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",30
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",27
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [71]:
y_pred_dt_bal = model_dt_bal.predict(X_test)

In [72]:
acc_dt_bal = accuracy_score(y_test, y_pred_dt_bal)
recall_dt_bal = recall_score(y_test, y_pred_dt_bal, average='macro')

print("Accuracy (Balanced DT):", acc_dt_bal)
print("Recall (Balanced DT - macro):", recall_dt_bal)

print("\nCLASSIFICATION REPORT (BALANCED DT)\n")
print(classification_report(y_test, y_pred_dt_bal))

print("\nCONFUSION MATRIX (BALANCED DT)\n")
print(confusion_matrix(y_test, y_pred_dt_bal))

Accuracy (Balanced DT): 0.90902
Recall (Balanced DT - macro): 0.8220740203709508

CLASSIFICATION REPORT (BALANCED DT)

              precision    recall  f1-score   support

           1       0.77      0.89      0.82      7722
           2       0.94      0.92      0.93    121413
           3       0.89      0.90      0.89     70264
           4       0.19      0.59      0.29       601

    accuracy                           0.91    200000
   macro avg       0.70      0.82      0.73    200000
weighted avg       0.92      0.91      0.91    200000


CONFUSION MATRIX (BALANCED DT)

[[  6841    740    130     11]
 [  1805 111476   7701    431]
 [   243   5811  63135   1075]
 [     3     48    198    352]]


In [73]:
precision_dt_bal=precision_score(y_test, y_pred_dt_bal, average='macro')

In [74]:
f1_dt_bal=f1_score(y_test, y_pred_dt_bal, average='macro')

### Comparison DT vs DT(balance)

In [75]:
from sklearn.metrics import precision_score, f1_score
precision_dt = precision_score(y_test, y_pred_dt, average='macro')
f1_dt = f1_score(y_test, y_pred_dt, average='macro')

precision_dt_bal = precision_score(y_test, y_pred_dt_bal, average='macro')
f1_dt_bal = f1_score(y_test, y_pred_dt_bal, average='macro')

comparison_full = pd.DataFrame({
    "Model": ["DT", "Balanced DT"],
    "Accuracy": [acc_dt, acc_dt_bal],
    "Precision": [precision_dt, precision_dt_bal],
    "Recall": [recall_dt, recall_dt_bal],
    "F1 Score": [f1_dt, f1_dt_bal]
})

print("\nFULL METRIC COMPARISON\n")
print(comparison_full)


FULL METRIC COMPARISON

         Model  Accuracy  Precision    Recall  F1 Score
0           DT  0.920785   0.759308  0.721647  0.736609
1  Balanced DT  0.909020   0.697242  0.822074  0.733079


# Random Forest

In [33]:
from sklearn.ensemble import RandomForestClassifier
model = RandomForestClassifier(n_estimators=100, random_state=42)

model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [34]:
y_pred = model.predict(X_test)

In [35]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.93056

Classification Report:

              precision    recall  f1-score   support

           1       0.87      0.83      0.85      7722
           2       0.94      0.95      0.95    121413
           3       0.92      0.91      0.92     70264
           4       0.61      0.02      0.04       601

    accuracy                           0.93    200000
   macro avg       0.84      0.68      0.69    200000
weighted avg       0.93      0.93      0.93    200000


Confusion Matrix:

[[  6441   1005    276      0]
 [   907 115444   5061      1]
 [    52   5990  64216      6]
 [     0     58    532     11]]


In [36]:
acc_rf=accuracy_score(y_test, y_pred)

In [51]:
precision_rf=precision_score(y_test, y_pred, average='macro')

In [38]:
recall_rf=recall_score(y_test, y_pred, average='macro')

In [53]:
f1_rf=f1_score(y_test, y_pred, average='macro')

#### Random Forest(Class balance)

In [40]:
model_balanced = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',  
    random_state=27
)

In [41]:
model_balanced.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap: bool, default=TrueWhether bootstrap samples are used when building trees. If False, thewhole dataset is used to build each tree.",True
,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric

In [42]:
y_pred_balanced = model_balanced.predict(X_test)

In [43]:
print("BALANCED MODEL RESULTS")
print("Accuracy:", accuracy_score(y_test, y_pred_balanced))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_balanced))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred_balanced))

BALANCED MODEL RESULTS
Accuracy: 0.92441

Classification Report:

              precision    recall  f1-score   support

           1       0.88      0.82      0.85      7722
           2       0.94      0.95      0.94    121413
           3       0.91      0.90      0.91     70264
           4       0.66      0.09      0.16       601

    accuracy                           0.92    200000
   macro avg       0.85      0.69      0.71    200000
weighted avg       0.92      0.92      0.92    200000


Confusion Matrix:

[[  6368   1101    253      0]
 [   831 114926   5652      4]
 [    41   6665  63534     24]
 [     0     64    483     54]]


In [44]:
acc_rf_bal=accuracy_score(y_test, y_pred_balanced)

In [54]:
precision_rf_bal=precision_score(y_test, y_pred_balanced, average='macro')

In [46]:
recall_rf_bal=recall_score(y_test, y_pred_balanced, average='macro')

In [47]:
f1_rf_bal=f1_score(y_test, y_pred_balanced, average='macro')

### Comparison RF vs RF(balance)

In [48]:
print(" MODEL COMPARISON ")

print("\nBaseline Accuracy:", accuracy_score(y_test, y_pred))
print("Balanced Accuracy:", accuracy_score(y_test, y_pred_balanced))

 MODEL COMPARISON 

Baseline Accuracy: 0.93056
Balanced Accuracy: 0.92441


In [49]:
print("\nRECALL COMPARISON")

print("Baseline Recall:", recall_score(y_test, y_pred, average=None))
print("Balanced Recall:", recall_score(y_test, y_pred_balanced, average=None))


RECALL COMPARISON
Baseline Recall: [0.83411033 0.95083723 0.91392463 0.01830283]
Balanced Recall: [0.82465682 0.9465708  0.90421838 0.08985025]


## Final Comparison

In [76]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "Balanced Random Forest", "Decision Tree", "Balanced Decision Tree"],
    "Accuracy": [acc_rf, acc_rf_bal, acc_dt, acc_dt_bal],
    "Precision": [precision_rf, precision_rf_bal, precision_dt, precision_dt_bal],
    "Recall": [recall_rf, recall_rf_bal, recall_dt, recall_dt_bal],
    "F1 Score": [f1_rf, f1_rf_bal, f1_dt, f1_dt_bal]
})

print("\nFINAL MODEL COMPARISON\n")
print(comparison.sort_values(by="Recall", ascending=False))


FINAL MODEL COMPARISON

                    Model  Accuracy  Precision    Recall  F1 Score
3  Balanced Decision Tree  0.909020   0.697242  0.822074  0.733079
2           Decision Tree  0.920785   0.759308  0.721647  0.736609
1  Balanced Random Forest  0.924410   0.845738  0.691324  0.714284
0           Random Forest  0.930560   0.835050  0.679294  0.687279
